# 📈 Model Comparisons and Evaluation

This notebook focuses on comparing different model architectures, selecting the most robust features using L1 regularization, and evaluating performance using Brier scores.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss
from xgboost import XGBClassifier

OUTPUT_PATH = '../output'
rs_data = pd.read_csv(f'{OUTPUT_PATH}/RegularDataModel.csv')
tourney_data = pd.read_csv(f'{OUTPUT_PATH}/TournamentDataModel.csv')

## 1. Feature Selection (L1 Regularization)

We use Lasso regression (`penalty='l1'`) to identify features that correlate most strongly with tournament outcomes. This helps minimize overfitting by zeroing out noisy features.

### Logic
The `LogisticRegressionCV` search finds the optimal inverse regularization strength ($C$) that minimizes the Brier Score. Features with non-zero coefficients at this $C$ are selected for the final ensemble.

In [ ]:
def feature_importance_plot(model, features):
    """
    Visualizes the magnitude of coefficients for the selected features.
    """
    coefs = pd.DataFrame({
        'Feature': features,
        'Coefficient': model.coef_[0]
    }).sort_values(by='Coefficient', ascending=False)

    plt.figure(figsize=(10, 8))
    sns.barplot(x='Coefficient', y='Feature', data=coefs)
    plt.title('Feature Importance via L1 Regularization')
    plt.show()

## 2. Classifier Comparison

During development, we tested three primary architectures:
1. **XGBoost**: Superior at capturing non-linear interactions and variance.
2. **Logistic Regression**: High reliability and less prone to overfitting on small datasets (crucial for the Women's tournament).
3. **Random Forest**: Aggregates variance and provides stability.

## 3. Probability Calibration

March Madness predictions require well-calibrated probabilities for the Brier Score. We compare **Platt Scaling (Sigmoid)** and **Isotonic Regression**.

- **Isotonic Regression**: Non-parametric, better for large datasets but requires more data to avoid overfitting.
- **Platt Scaling**: Parametric (Logistic), safer for smaller cohorts like the Women's historical data.

For the final 2026 model, we opted for **Isotonic Regression** due to the size of our combined (Regular Season + Tournament) training set.

In [ ]:
def plot_calibration_curve(y_true, y_prob, name):
    """
    Plots the reliability diagram (calibration curve).
    """
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10)
    plt.plot(prob_pred, prob_true, marker='o', label=name)
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('Mean Predicted Probability')
    plt.ylabel('Fraction of Positives')
    plt.legend()
    plt.title('Reliability Diagram')
    plt.show()

## 4. Final Ensemble Strategy

We use a **Soft Voting Ensemble** to combine the strengths of the three models. Weights are adjusted by league:
- **Men's**: Weighted towards XGBoost (2:1:1) to capture high-variance upsets.
- **Women's**: Weighted towards Logistic Regression (1:2:1) due to the higher predictability (lower variance) of historical seeds in the women's tournament.